In [1]:
import requests
import re
import json
import pandas as pd
import time
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import os

In [2]:
# 1. Obtener el HTML de la página del ranking
load_dotenv()
url = os.getenv("TARGET_URL_UNEMPLOYMENT_RATE")
BASE_URL = os.getenv("BASE_URL")

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
response.raise_for_status()  # Lanza error si la petición falla

In [3]:
# 2. Parsear HTML con BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

In [4]:
# 3. Buscar la tabla principal y extraer los links
table = soup.find("table")
country_links = []


if table:
    rows = table.find_all("tr")
    for row in rows:
        cell = row.find("td")
        if cell and cell.a:
            relative_link = cell.a["href"]
            full_url = f"{BASE_URL}{relative_link}"
            country_links.append(full_url)

# Convertir la lista en DataFrame
df_links = pd.DataFrame(country_links, columns=["url"])

# Guardar como archivo Parquet
df_links.to_parquet("../../data/country_unemployment_links.parquet", index=False)

In [5]:
# 4. Mostrar resultado
print("Links encontrados:", df_links.size)

Links encontrados: 106


In [6]:
# configuracion necesaria para descargar datos

master_df = pd.DataFrame()

pattern = r'arrayToDataTable\((\[\[.*?\]\])\)'

sleep_duration = 5

In [7]:
# Cargar los países válidos desde el archivo parquet
df_countries = pd.read_parquet("../../data/countries_with_hpi.parquet")
valid_countries = set(df_countries["country"].unique())

In [8]:
start_time = time.time()
for url in country_links:
    # Extraer país desde el URL
    country = urlparse(url).path.strip("/").split("/")[0]

    # Verificar si el país está en la lista válida
    if country not in valid_countries:
        print(f"⏭️ País {country} no está en la lista, se omite.")
        continue

    try:
        response = requests.get(url, headers=headers)
        html = response.text

        match = re.search(pattern, html, re.DOTALL)
        if match:
            data_str = match.group(1)
            data = json.loads(data_str)

            df = pd.DataFrame(data[1:], columns=data[0])
            df.columns = ["Period", "Value"]
            df["Country"] = country

            df_pivot = df.pivot(index="Period", columns="Country", values="Value")
            master_df = pd.merge(master_df, df_pivot, how="outer", left_index=True, right_index=True)

            print(f"✅ Datos extraídos correctamente para {country}")
        else:
            print(f"⚠️ No se encontraron datos en {url}")
    except Exception as e:
        print(f"❌ Error procesando {url}: {e}")
    
    time.sleep(sleep_duration)

end_time = time.time()
total_sleep_time = len(valid_countries) * sleep_duration
effective_time = end_time - start_time - total_sleep_time

print(f"⏳ Tiempo total (incluyendo esperas): {end_time - start_time:.2f} s")
print(f"😴 Tiempo en espera: {total_sleep_time:.2f} s")
print(f"⚡ Tiempo efectivo de ejecución: {effective_time:.2f} s")

⏭️ País Albania no está en la lista, se omite.
⏭️ País Algeria no está en la lista, se omite.
⏭️ País Andorra no está en la lista, se omite.
⏭️ País Angola no está en la lista, se omite.
⏭️ País Argentina no está en la lista, se omite.
⏭️ País Armenia no está en la lista, se omite.
✅ Datos extraídos correctamente para Australia
✅ Datos extraídos correctamente para Austria
⏭️ País Azerbaijan no está en la lista, se omite.
⏭️ País Barbados no está en la lista, se omite.
⏭️ País Belarus no está en la lista, se omite.
✅ Datos extraídos correctamente para Belgium
⏭️ País Bolivia no está en la lista, se omite.
⏭️ País Bosnia-and-Herzegovina no está en la lista, se omite.
✅ Datos extraídos correctamente para Brazil
✅ Datos extraídos correctamente para Bulgaria
✅ Datos extraídos correctamente para Canada
✅ Datos extraídos correctamente para Chile
✅ Datos extraídos correctamente para China
✅ Datos extraídos correctamente para Colombia
⏭️ País Costa-Rica no está en la lista, se omite.
✅ Datos ex

In [9]:
master_df.tail()

,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,Croatia,...,Slovenia,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom
Period,,,,,,,,,,,,,,,,,,,,,
Sep 2020,6.72,6.0,6.4,14.9,5.8,8.4,12.35,NaN,15.77,7.6,...,4.9,NaN,3.6,16.4,8.4,5.2,NaN,12.4,7.8,5.2
Sep 2021,4.56,4.6,6.3,12.7,4.3,6.5,8.41,NaN,11.96,6.4,...,4.5,NaN,2.7,14.4,8.4,4.9,NaN,11.1,4.7,4.3
Sep 2022,3.48,4.9,5.6,8.7,3.4,4.8,8.04,NaN,10.75,6.4,...,3.8,NaN,2.4,13.0,6.5,4.0,NaN,9.9,3.5,3.8
Sep 2023,3.45,5.1,5.6,7.7,3.9,5.2,8.91,NaN,9.25,5.6,...,3.7,NaN,2.3,12.1,7.7,4.4,NaN,8.9,3.8,4.0
Sep 2024,3.89,5.4,6.0,6.4,3.5,6.0,8.74,NaN,9.13,4.6,...,4.3,NaN,2.1,11.1,8.2,4.8,NaN,8.6,4.1,4.3


In [10]:
# obtain paraguay url
template_url = country_links[0]
country = "Paraguay"

# Separar por "/"
parts = template_url.strip("/").split("/")

# Reemplazar el país (posición 3: después del dominio)

parts[3] = country

# Reconstruir la URL
custom_url = "/".join(parts) + "/"

In [11]:
# URL específica para Paraguay
response = requests.get(custom_url, headers=headers)
html = response.text

# Buscar el bloque JSON dentro del HTML
match = re.search(pattern, html, re.DOTALL)
if match:
    data_str = match.group(1)
    data = json.loads(data_str)

    # Crear DataFrame con los datos
    df = pd.DataFrame(data[1:], columns=data[0])
    df.columns = ["Period", "Value"]
    df["Country"] = "Paraguay"  # Asignar explícitamente

    # Pivotear para formato ancho (opcional)
    df_paraguay = df.pivot(index="Period", columns="Country", values="Value")

    print("✅ Datos de Paraguay extraídos correctamente")
else:
    print("⚠️ No se encontraron datos en el HTML para Paraguay")

✅ Datos de Paraguay extraídos correctamente


In [12]:
df_paraguay.tail()

Country,Paraguay
Period,
Q4 2020,7.17
Q4 2021,6.81
Q4 2022,5.80
Q4 2023,5.23
Q4 2024,4.55


In [13]:
# Unir ambos DataFrames por el índice 'Period', preservando todos los datos
master_df = pd.merge(master_df, df_paraguay, how="outer", left_index=True, right_index=True)

In [14]:
master_df.head()

,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,Croatia,...,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay
Period,,,,,,,,,,,,,,,,,,,,,
Apr 1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.2,1.6,NaN
Apr 1961,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,1.3,NaN
Apr 1962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.6,1.7,NaN
Apr 1963,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.7,2.3,NaN
Apr 1964,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.3,1.6,NaN


In [15]:
unique_periods = pd.Series(master_df.index.astype(str)).unique()
print(unique_periods)

['Apr 1960' 'Apr 1961' 'Apr 1962' 'Apr 1963' 'Apr 1964' 'Apr 1965'
 'Apr 1966' 'Apr 1967' 'Apr 1968' 'Apr 1969' 'Apr 1970' 'Apr 1971'
 'Apr 1972' 'Apr 1973' 'Apr 1974' 'Apr 1975' 'Apr 1976' 'Apr 1977'
 'Apr 1978' 'Apr 1979' 'Apr 1980' 'Apr 1981' 'Apr 1982' 'Apr 1983'
 'Apr 1984' 'Apr 1985' 'Apr 1986' 'Apr 1987' 'Apr 1988' 'Apr 1989'
 'Apr 1990' 'Apr 1991' 'Apr 1992' 'Apr 1993' 'Apr 1994' 'Apr 1995'
 'Apr 1996' 'Apr 1997' 'Apr 1998' 'Apr 1999' 'Apr 2000' 'Apr 2001'
 'Apr 2002' 'Apr 2003' 'Apr 2004' 'Apr 2005' 'Apr 2006' 'Apr 2007'
 'Apr 2008' 'Apr 2009' 'Apr 2010' 'Apr 2011' 'Apr 2012' 'Apr 2013'
 'Apr 2014' 'Apr 2015' 'Apr 2016' 'Apr 2017' 'Apr 2018' 'Apr 2019'
 'Apr 2020' 'Apr 2021' 'Apr 2022' 'Apr 2023' 'Apr 2024' 'Apr 2025'
 'Aug 1960' 'Aug 1961' 'Aug 1962' 'Aug 1963' 'Aug 1964' 'Aug 1965'
 'Aug 1966' 'Aug 1967' 'Aug 1968' 'Aug 1969' 'Aug 1970' 'Aug 1971'
 'Aug 1972' 'Aug 1973' 'Aug 1974' 'Aug 1975' 'Aug 1976' 'Aug 1977'
 'Aug 1978' 'Aug 1979' 'Aug 1980' 'Aug 1981' 'Aug 1982' 'Aug 1

In [16]:
# Reset index to expose 'Period'
master_df2 = master_df.copy()
master_df2 = master_df.reset_index()

# Month to Quarter mapping
month_to_quarter = {
    "Jan": "Q1", "Feb": "Q1", "Mar": "Q1",
    "Apr": "Q2", "May": "Q2", "Jun": "Q2", "June": "Q2",
    "Jul": "Q3", "Aug": "Q3", "Sep": "Q3",
    "Oct": "Q4", "Nov": "Q4", "Dec": "Q4"
}

# Function to standardize each Period entry
def standardize_period(period):
    period = str(period).strip()

    # Match "Month Year" format
    match_month = re.match(r"^([A-Za-z]{3,4}) (\d{4})$", period)
    if match_month:
        month, year = match_month.groups()
        quarter = month_to_quarter.get(month)
        if quarter:
            return f"{quarter} {year}"

    # Match "Quarter Year" format
    match_quarter = re.match(r"^(Q[1-4]) (\d{4})$", period)
    if match_quarter:
        quarter, year = match_quarter.groups()
        return f"{quarter} {year}"

    # If no match, return None or original for inspection
    return None

In [17]:
# Apply the function
master_df2["Standardized_Period"] = master_df2["Period"].apply(standardize_period)

In [18]:
# Drop rows where standardization failed
master_df2 = master_df2[master_df2["Standardized_Period"].notna()].copy()

In [19]:
# Split standardized period into Quarter and Year
master_df2[["Quarter", "Year"]] = master_df2["Standardized_Period"].str.extract(r"(Q[1-4]) (\d{4})")
master_df2["Year"] = master_df2["Year"].astype(int)

In [20]:
# Reorder columns
cols = ["Quarter", "Year"] + [col for col in master_df2.columns if col not in ["Period", "Month", "Quarter", "Year", "Standardized_Period"]]
master_df2 = master_df2[cols]

In [21]:
# Sort by Year and Quarter
master_df2 = master_df2.sort_values(by=["Year", "Quarter"]).reset_index(drop=True)

In [22]:
master_df2.head()

,Quarter,Year,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,...,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay
0,Q1,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.8,1.6,NaN
1,Q1,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.2,1.7,NaN
2,Q1,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.4,1.6,NaN
3,Q2,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.2,1.6,NaN
4,Q2,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.4,1.5,NaN


In [23]:
master_df2["Quarter"].unique().tolist()

['Q1', 'Q2', 'Q3', 'Q4']

In [24]:
master_df2[master_df2["Quarter"].isnull()]

,Quarter,Year,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,...,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay


In [25]:
def convert_quarter_to_int(df, quarter_col="Quarter"):
    """
    Converts quarter strings like 'Q1' to integers (1–4) in the specified column.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame.
        quarter_col (str): The name of the column containing quarter strings.
    
    Returns:
        pd.DataFrame: A copy of the DataFrame with the quarter column converted to integers.
    """
    quarter_map = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}
    df = df.copy()
    df[quarter_col] = df[quarter_col].map(quarter_map)
    return df

In [26]:
master_df3 = master_df2.copy()

In [27]:
master_df3 = convert_quarter_to_int(master_df3)
master_df3.head()

,Quarter,Year,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,...,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay
0,1,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.8,1.6,NaN
1,1,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.2,1.7,NaN
2,1,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.4,1.6,NaN
3,2,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.2,1.6,NaN
4,2,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.4,1.5,NaN


In [28]:
master_df3["Quarter"].unique().tolist()

[1, 2, 3, 4]

In [29]:
def convert_year_to_int(df, year_col="Year"):
    """
    Converts the specified year column to integers, handling strings and other types.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame.
        year_col (str): The name of the column containing year values.
    
    Returns:
        pd.DataFrame: A copy of the DataFrame with the year column as integers.
    """
    df = df.copy()
    df[year_col] = pd.to_numeric(df[year_col], errors="coerce").astype("Int64")
    return df

In [30]:
master_df3 = convert_year_to_int(master_df3)
master_df3.head()

,Quarter,Year,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,...,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay
0,1,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.8,1.6,NaN
1,1,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.2,1.7,NaN
2,1,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.4,1.6,NaN
3,2,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.2,1.6,NaN
4,2,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.4,1.5,NaN


In [31]:
master_df3["Year"].unique().tolist()

[1960,
 1961,
 1962,
 1963,
 1964,
 1965,
 1966,
 1967,
 1968,
 1969,
 1970,
 1971,
 1972,
 1973,
 1974,
 1975,
 1976,
 1977,
 1978,
 1979,
 1980,
 1981,
 1982,
 1983,
 1984,
 1985,
 1986,
 1987,
 1988,
 1989,
 1990,
 1991,
 1992,
 1993,
 1994,
 1995,
 1996,
 1997,
 1998,
 1999,
 2000,
 2001,
 2002,
 2003,
 2004,
 2005,
 2006,
 2007,
 2008,
 2009,
 2010,
 2011,
 2012,
 2013,
 2014,
 2015,
 2016,
 2017,
 2018,
 2019,
 2020,
 2021,
 2022,
 2023,
 2024,
 2025]

In [33]:
# Guardar como archivo Parquet
master_df3.to_parquet("../../data/unemployment_rates_values.parquet", index=False)